# How to ensure reproducibility of the results

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ing-bank/probatus/blob/master/docs/howto/reproducibility.ipynb)

This page explains how to ensure full reproducibility in your analyses using `probatus`.

Reproducibility is influenced by two factors:
- The inputs provided to `probatus` modules.
- The `random_state` setting of `probatus` modules.

The sections below describe how to control these aspects.

## Inputs of probatus Modules

`probatus` modules accept various parameters. Below, we highlight the most common ones.

### Static Dataset

The dataset is a critical component. Ensure that the dataset remains unchanged throughout the analysis.

The code snippet below demonstrates random data preparation. In scikit-learn, setting the `random_state` parameter guarantees reproducibility. Although you may use a different dataset in your projects, always ensure that the input data is static.

In [1]:
%%capture
!pip install probatus

In [2]:
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=100, n_features=10, n_redundant=1, n_informative=3, random_state=42)

### Static Data Splits

Ensure that data splits remain consistent throughout your experiments. For instance, when using scikit-learn's `train_test_split`, set the `random_state` parameter to guarantee reproducibility.

Additionally, consider the `cv` parameter, which determines the cross-validation fold settings. If `cv` is an integer, `probatus` manages reproducibility via its own `random_state`. However, if you provide a custom cv generator, you must explicitly set its `random_state`.

Below are examples of static data splits.

In [3]:
from sklearn.model_selection import StratifiedKFold, train_test_split

# Perform a static train/test split for reproducibility.
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# Define static cross-validation settings:
cv1 = 5  # Number of folds for basic cross-validation.
cv2 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

### Static Classifier

Most probatus modules accept classifiers directly. For an unfitted classifier, simply setting the `random_state` is sufficient. If the classifier is pre-fitted, ensure that its training process is reproducible.

In [4]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(random_state=42)

### Static Search CV for Hyperparameter Tuning

Some modules, such as `ShapRFECV`, support model optimization. To ensure reproducibility, set the `random_state` for these classes. This guarantees that each optimization round explores the same parameter permutations. If the search space itself involves randomness, make sure to set the `random_state` there as well.

In [5]:
from sklearn.model_selection import RandomizedSearchCV

# Define the hyperparameter grid for tuning.
param_grid = {
    "n_estimators": [5, 7, 10],
    "max_leaf_nodes": [3, 5, 7, 10],
}

# Initialize RandomizedSearchCV with a single iteration and a fixed random state.
search = RandomizedSearchCV(model, param_grid, n_iter=1, random_state=42)

### Other Sources of Randomness

Before running `probatus` modules, review all inputs to identify potential sources of randomness. If randomness is present, one way to ensure reproducibility is by setting a global random seed at the beginning of the code.

In [6]:
# Optional step
import numpy as np

np.random.seed(42)

## Reproducibility in probatus

`probatus` modules allow you to set the `random_state`, ensuring a consistent execution flow. As long as this is set and all other inputs remain stable, you can achieve fully reproducible results across runs.

In [7]:
from probatus.features import ShapRFECV

shap_elimination = ShapRFECV(model=search, step=0.2, cv=cv2, scoring="roc_auc", n_jobs=3, random_state=42)
report = shap_elimination.fit_compute(X, y)

report[["num_features", "eliminated_features", "val_metric_mean"]]

Feature Elimination:   0%|          | 0/9 [00:00<?, ?it/s]

,num_features,eliminated_features,val_metric_mean
1,10,"[4, 9]",0.928899
2,8,[7],0.922929
3,7,[6],0.925818
4,6,[2],0.939970
5,5,[5],0.918919
6,4,[8],0.952990
7,3,[0],0.939899
8,2,[1],0.931788
9,1,[],0.889717
